🔄 Script de Retreinamento Seguro — Safra 2025 (Com Checkpoints)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
from google.colab import drive

# 1. Montar Drive (Obrigatório)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CONFIGURAÇÕES DA SAFRA 2025 ---
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'
SAFRA_INICIO = '2024-12-01'
SAFRA_FIM = '2025-03-31'
DATA_1_DESC = 'janela inicial da safra (dezembro/2024)'
DATA_2_DESC = 'janela final da safra (março/2025)'

# Busca primeiro um TFRecord MASSIVE da safra-alvo e, se não existir,
# usa qualquer TFRecord da safra-alvo. Assim evitamos treinar 2025 com dados
# antigos por engano.
padroes_busca = [
    f'*{SAFRA_ALVO}*MASSIVE*tfrecord*',
    f'*MASSIVE*{SAFRA_ALVO}*tfrecord*',
    f'*{SAFRA_ALVO}*tfrecord*',
]

busca = []
for padrao in padroes_busca:
    busca = glob.glob(os.path.join(pasta_base, padrao))
    if busca:
        busca.sort(key=os.path.getmtime, reverse=True)
        break

if not busca:
    raise FileNotFoundError(
        f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}. '
        'Confira se o arquivo contém 2025 no nome.'
    )

caminho_arquivo = busca[0]
print(f"📂 Lendo dados da safra {SAFRA_ALVO}: {caminho_arquivo}")
print(f"🗓️ Janela fenológica configurada: {SAFRA_INICIO} até {SAFRA_FIM}")

# Parâmetros
KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS = 40
# _1 representa a janela inicial (dez/2024) e _2 representa a janela final (mar/2025).
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
USAR_FENOLOGIA = True
USAR_VARIACAO_PAISAGEM = True
USAR_AUGE_VIGOR = True
LIMIAR_ALTO_VIGOR_NDVI = 0.55
FENOLOGIA_BANDS = [
    'DELTA_NDVI',          # crescimento/queda entre as duas datas da safra
    'ABS_DELTA_NDVI',      # amplitude fenológica: cerrado tende a variar menos que plantio
    'NDVI_MAX',            # pico de vigor vegetativo
    'NDVI_MEAN',           # vigor médio no período
    'PLANTIO_SIGNAL',      # aumento positivo de NDVI, típico de lavoura em implantação
    'CERRADO_STABILITY',   # vegetação verde e estável no período da safra
]
AUGE_VIGOR_BANDS = [
    'VIGOR_MAX_NDVI',         # maior NDVI observado na janela dez/2024-mar/2025
    'VIGOR_PEAK_TIMING',      # 0=início da janela; 1=fim da janela
    'VIGOR_PEAK_CONFIDENCE',  # diferença absoluta entre NDVI_1 e NDVI_2
    'VIGOR_ALTO_MASK',        # máscara de alto vigor pelo limiar definido
]
PAISAGEM_BANDS = [
    'COLHEITA_SIGNAL',       # queda de NDVI: possível colheita/senescência
    'SOLO_EXPOSTO_1',        # solo exposto na janela inicial
    'SOLO_EXPOSTO_2',        # solo exposto na janela final
    'SOLO_EXPOSTO_AUMENTO',  # aumento de solo exposto entre as datas
    'MUDANCA_PAISAGEM',      # variação média em R, NIR e NDVI
    'ALERTA_MUDANCA',        # alerta regra: colheita, solo exposto ou mudança forte
]
FEATURE_BANDS = (
    INPUT_BANDS
    + (FENOLOGIA_BANDS if USAR_FENOLOGIA else [])
    + (AUGE_VIGOR_BANDS if USAR_AUGE_VIGOR else [])
    + (PAISAGEM_BANDS if USAR_VARIACAO_PAISAGEM else [])
)

# --- PIPELINE DE DADOS (Rápido) ---
def parse_and_process(example_proto):
    features_dict = {
        band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]
    }
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)

    if USAR_FENOLOGIA:
        r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
        delta_ndvi = ndvi2 - ndvi1
        abs_delta_ndvi = tf.abs(delta_ndvi)
        ndvi_max = tf.maximum(ndvi1, ndvi2)
        ndvi_mean = (ndvi1 + ndvi2) / 2.0
        plantio_signal = tf.nn.relu(delta_ndvi)
        cerrado_stability = ndvi_mean * (1.0 - tf.clip_by_value(abs_delta_ndvi, 0.0, 1.0))
        fenologia_stacked = tf.concat([
            delta_ndvi,
            abs_delta_ndvi,
            ndvi_max,
            ndvi_mean,
            plantio_signal,
            cerrado_stability,
        ], axis=-1)
        image_stacked = tf.concat([image_stacked, fenologia_stacked], axis=-1)

    if USAR_AUGE_VIGOR:
        r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
        vigor_max_ndvi = tf.maximum(ndvi1, ndvi2)
        vigor_peak_timing = tf.cast(ndvi2 >= ndvi1, tf.float32)
        vigor_peak_confidence = tf.abs(ndvi2 - ndvi1)
        vigor_alto_mask = tf.cast(vigor_max_ndvi >= LIMIAR_ALTO_VIGOR_NDVI, tf.float32)
        auge_vigor_stacked = tf.concat([
            vigor_max_ndvi,
            vigor_peak_timing,
            vigor_peak_confidence,
            vigor_alto_mask,
        ], axis=-1)
        image_stacked = tf.concat([image_stacked, auge_vigor_stacked], axis=-1)

    if USAR_VARIACAO_PAISAGEM:
        r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
        colheita_signal = tf.nn.relu(ndvi1 - ndvi2)
        solo_exposto_1 = tf.cast(ndvi1 < 0.25, tf.float32)
        solo_exposto_2 = tf.cast(ndvi2 < 0.25, tf.float32)
        solo_exposto_aumento = tf.nn.relu(solo_exposto_2 - solo_exposto_1)
        mudanca_paisagem = (tf.abs(r2 - r1) + tf.abs(nir2 - nir1) + tf.abs(ndvi2 - ndvi1)) / 3.0
        alerta_mudanca = tf.cast(
            (colheita_signal > 0.20) | (solo_exposto_aumento > 0.50) | (mudanca_paisagem > 0.20),
            tf.float32,
        )
        paisagem_stacked = tf.concat([
            colheita_signal,
            solo_exposto_1,
            solo_exposto_2,
            solo_exposto_aumento,
            mudanca_paisagem,
            alerta_mudanca,
        ], axis=-1)
        image_stacked = tf.concat([image_stacked, paisagem_stacked], axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Contagem rápida para não dar erro de tamanho
print("🔢 Verificando tamanho do arquivo...")
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f"✅ Total de amostras: {N_REAL}")

N_TRAIN = int(N_REAL * 0.8)
full_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(parse_and_process)

train_ds = full_dataset.take(N_TRAIN).cache().shuffle(N_TRAIN).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- MODELO U-NET ---
def build_unet(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Camadas (Encoder)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs); p1 = layers.MaxPooling2D()(c1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1); p2 = layers.MaxPooling2D()(c2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2); p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)

    # Decoder
    u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)

    u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)

    u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c7)
    return models.Model(inputs=[inputs], outputs=[outputs])

print(f"🌱 Usando {len(FEATURE_BANDS)} bandas de entrada: {FEATURE_BANDS}")
model = build_unet((KERNEL_SIZE, KERNEL_SIZE, len(FEATURE_BANDS)))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# --- 🛡️ SALVAMENTO AUTOMÁTICO (SEGURANÇA) ---
# Salva um backup no Drive a cada época para a safra-alvo.
checkpoint_path = os.path.join(pasta_base, f'Modelo_Checkpoint_Jussara_{SAFRA_ALVO}.keras')
checkpoint_cb = callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=False, # Salva sempre o último estado
    verbose=1
)

print(f"🔥 Iniciando retreinamento da safra {SAFRA_ALVO} (salvamento automático no Drive)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb] # <--- Aqui está a segurança
)

# Salvamento Final Definitivo
final_path = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{SAFRA_ALVO}_FINAL_v2.keras')
model.save(final_path)
print(f"✅ SUCESSO! Modelo final da safra {SAFRA_ALVO} salvo em: {final_path}")


## Célula nova — Treinar solo exposto após colheita

Esta etapa treina um modelo auxiliar para mapear áreas com sinal de colheita e solo exposto na janela final da safra. Ela usa uma máscara-alvo derivada de regras (`NDVI_1 - NDVI_2`, `NDVI_2` baixo e aumento de reflectância no vermelho) para iniciar o mapeamento mesmo antes de termos um rótulo manual específico de solo exposto.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
from google.colab import drive

# 1. Montar Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("--- TREINANDO MODELO AUXILIAR: SOLO EXPOSTO APÓS COLHEITA ---")

# Configurações da safra
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'
SAFRA_INICIO = '2024-12-01'
SAFRA_FIM = '2025-03-31'

padroes_busca = [
    f'*{SAFRA_ALVO}*MASSIVE*tfrecord*',
    f'*MASSIVE*{SAFRA_ALVO}*tfrecord*',
    f'*{SAFRA_ALVO}*tfrecord*',
]

busca = []
for padrao in padroes_busca:
    busca = glob.glob(os.path.join(pasta_base, padrao))
    if busca:
        busca.sort(key=os.path.getmtime, reverse=True)
        break

if not busca:
    raise FileNotFoundError(
        f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}. '
        'Confira se o arquivo contém 2025 no nome.'
    )

caminho_arquivo = busca[0]
print(f"📂 Lendo dados para solo exposto pós-colheita: {caminho_arquivo}")
print(f"🗓️ Janela usada: {SAFRA_INICIO} até {SAFRA_FIM}")

# Parâmetros
KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS_SOLO_EXPOSTO = 40
LEARNING_RATE_SOLO = 1e-4
PATIENCE_SOLO = 8

INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
SOLO_FEATURE_BANDS = INPUT_BANDS + [
    'DELTA_NDVI_COLHEITA',
    'SOLO_EXPOSTO_FINAL',
    'AUMENTO_SOLO_EXPOSTO',
    'VERMELHO_FINAL_ALTO',
]

# Limiar inicial para criar uma máscara-alvo automática de solo exposto após colheita.
# Ajuste estes valores depois de comparar com amostras visuais e validação de campo.
LIMIAR_QUEDA_NDVI_COLHEITA = 0.20
LIMIAR_NDVI_SOLO_EXPOSTO = 0.25
LIMIAR_AUMENTO_R = 0.05


def parse_solo_exposto_pos_colheita(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS}
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)

    r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
    queda_ndvi = tf.nn.relu(ndvi1 - ndvi2)
    solo_exposto_final = tf.cast(ndvi2 < LIMIAR_NDVI_SOLO_EXPOSTO, tf.float32)
    aumento_solo_exposto = tf.cast((ndvi1 >= LIMIAR_NDVI_SOLO_EXPOSTO) & (ndvi2 < LIMIAR_NDVI_SOLO_EXPOSTO), tf.float32)
    vermelho_final_alto = tf.cast((r2 - r1) > LIMIAR_AUMENTO_R, tf.float32)

    # Máscara-alvo: caiu o vigor, terminou com NDVI baixo e aumentou sinal de solo/refletância.
    label_solo_pos_colheita = tf.cast(
        (queda_ndvi > LIMIAR_QUEDA_NDVI_COLHEITA)
        & (solo_exposto_final > 0.5)
        & ((aumento_solo_exposto > 0.5) | (vermelho_final_alto > 0.5)),
        tf.float32,
    )

    image_stacked = tf.concat([
        r1, nir1, ndvi1, r2, nir2, ndvi2,
        queda_ndvi,
        solo_exposto_final,
        aumento_solo_exposto,
        vermelho_final_alto,
    ], axis=-1)

    return image_stacked, label_solo_pos_colheita


def build_unet_solo(input_shape):
    inputs = layers.Input(shape=input_shape)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)

    u4 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c3)
    u4 = layers.concatenate([u4, c2])
    c4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u4)

    u5 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c1])
    c5 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u5)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c5)
    return models.Model(inputs=inputs, outputs=outputs)


def dice_coef_solo(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def bce_dice_loss_solo(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + (1.0 - dice_coef_solo(y_true, y_pred))


print("🔢 Verificando tamanho do arquivo...")
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f"✅ Total de amostras: {N_REAL}")

N_TRAIN = int(N_REAL * 0.8)
full_dataset_solo = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(
    parse_solo_exposto_pos_colheita,
    num_parallel_calls=tf.data.AUTOTUNE,
)

train_ds_solo = (
    full_dataset_solo
    .take(N_TRAIN)
    .cache()
    .shuffle(N_TRAIN)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds_solo = full_dataset_solo.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"🟫 Treinando com {len(SOLO_FEATURE_BANDS)} bandas: {SOLO_FEATURE_BANDS}")
modelo_solo = build_unet_solo((KERNEL_SIZE, KERNEL_SIZE, len(SOLO_FEATURE_BANDS)))
modelo_solo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_SOLO),
    loss=bce_dice_loss_solo,
    metrics=[dice_coef_solo, tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')],
)

checkpoint_solo_path = os.path.join(pasta_base, f'Modelo_Checkpoint_Solo_Exposto_Pos_Colheita_{SAFRA_ALVO}.keras')
checkpoint_solo_cb = callbacks.ModelCheckpoint(
    filepath=checkpoint_solo_path,
    monitor='val_dice_coef_solo',
    mode='max',
    save_best_only=True,
    verbose=1,
)
early_stop_solo_cb = callbacks.EarlyStopping(
    monitor='val_dice_coef_solo',
    mode='max',
    patience=PATIENCE_SOLO,
    restore_best_weights=True,
    verbose=1,
)

history_solo = modelo_solo.fit(
    train_ds_solo,
    validation_data=val_ds_solo,
    epochs=EPOCHS_SOLO_EXPOSTO,
    callbacks=[checkpoint_solo_cb, early_stop_solo_cb],
)

final_solo_path = os.path.join(pasta_base, f'Modelo_Solo_Exposto_Pos_Colheita_{SAFRA_ALVO}_FINAL.keras')
modelo_solo.save(final_solo_path)
print(f"✅ Modelo de solo exposto pós-colheita salvo em: {final_solo_path}")


## Célula nova — Treinamento específico dos pivôs

Esta célula treina um modelo auxiliar focado somente em encontrar pivôs usando o `label_chip` como máscara positiva. Ela fica separada do treinamento de solo exposto para permitir avaliar pivô primeiro e depois cruzar com fenologia, colheita e alerta.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import glob
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("--- TREINANDO MODELO AUXILIAR: PIVÔS ---")

pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'
SAFRA_INICIO = '2024-12-01'
SAFRA_FIM = '2025-03-31'

padroes_busca = [
    f'*{SAFRA_ALVO}*MASSIVE*tfrecord*',
    f'*MASSIVE*{SAFRA_ALVO}*tfrecord*',
    f'*{SAFRA_ALVO}*tfrecord*',
]

busca = []
for padrao in padroes_busca:
    busca = glob.glob(os.path.join(pasta_base, padrao))
    if busca:
        busca.sort(key=os.path.getmtime, reverse=True)
        break

if not busca:
    raise FileNotFoundError(f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}.')

caminho_arquivo = busca[0]
print(f"📂 Lendo dados para treinamento de pivôs: {caminho_arquivo}")

KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS_PIVOS = 60
LEARNING_RATE_PIVOS = 1e-4
PATIENCE_PIVOS = 10

INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
PIVO_FEATURE_BANDS = INPUT_BANDS + [
    'DELTA_NDVI',
    'ABS_DELTA_NDVI',
    'NDVI_MAX',
    'NDVI_MEAN',
    'COLHEITA_SIGNAL',
    'SOLO_EXPOSTO_2',
]


def parse_pivos(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)

    r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
    delta_ndvi = ndvi2 - ndvi1
    abs_delta_ndvi = tf.abs(delta_ndvi)
    ndvi_max = tf.maximum(ndvi1, ndvi2)
    ndvi_mean = (ndvi1 + ndvi2) / 2.0
    colheita_signal = tf.nn.relu(ndvi1 - ndvi2)
    solo_exposto_2 = tf.cast(ndvi2 < 0.25, tf.float32)

    image_stacked = tf.concat([
        r1, nir1, ndvi1, r2, nir2, ndvi2,
        delta_ndvi, abs_delta_ndvi, ndvi_max, ndvi_mean,
        colheita_signal, solo_exposto_2,
    ], axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image_stacked, lbl


def augmentar_pivo(image, label):
    stacked = tf.concat([image, label], axis=-1)
    if tf.random.uniform(()) > 0.5:
        stacked = tf.image.flip_left_right(stacked)
    if tf.random.uniform(()) > 0.5:
        stacked = tf.image.flip_up_down(stacked)
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    stacked = tf.image.rot90(stacked, k=k)
    return stacked[:, :, :-1], stacked[:, :, -1:]


def build_unet_pivos(input_shape):
    inputs = layers.Input(shape=input_shape)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    u4 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c3)
    u4 = layers.concatenate([u4, c2])
    c4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    u5 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c1])
    c5 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u5)
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c5)
    return models.Model(inputs=inputs, outputs=outputs)


def dice_coef_pivos(y_true, y_pred, smooth=1.0):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def bce_dice_loss_pivos(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + (1.0 - dice_coef_pivos(y_true, y_pred))


raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
N_TRAIN = int(N_REAL * 0.8)
print(f"✅ Total de amostras: {N_REAL} | Treino: {N_TRAIN} | Validação: {N_REAL - N_TRAIN}")

full_dataset_pivos = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(
    parse_pivos,
    num_parallel_calls=tf.data.AUTOTUNE,
)
train_ds_pivos = (
    full_dataset_pivos
    .take(N_TRAIN)
    .cache()
    .shuffle(N_TRAIN)
    .map(augmentar_pivo, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds_pivos = full_dataset_pivos.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

modelo_pivos = build_unet_pivos((KERNEL_SIZE, KERNEL_SIZE, len(PIVO_FEATURE_BANDS)))
modelo_pivos.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_PIVOS),
    loss=bce_dice_loss_pivos,
    metrics=[dice_coef_pivos, tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')],
)

checkpoint_pivos_path = os.path.join(pasta_base, f'Modelo_Checkpoint_Pivos_{SAFRA_ALVO}.keras')
checkpoint_pivos_cb = callbacks.ModelCheckpoint(
    filepath=checkpoint_pivos_path,
    monitor='val_dice_coef_pivos',
    mode='max',
    save_best_only=True,
    verbose=1,
)
early_stop_pivos_cb = callbacks.EarlyStopping(
    monitor='val_dice_coef_pivos',
    mode='max',
    patience=PATIENCE_PIVOS,
    restore_best_weights=True,
    verbose=1,
)

print(f"⭕ Treinando pivôs com {len(PIVO_FEATURE_BANDS)} bandas: {PIVO_FEATURE_BANDS}")
history_pivos = modelo_pivos.fit(
    train_ds_pivos,
    validation_data=val_ds_pivos,
    epochs=EPOCHS_PIVOS,
    callbacks=[checkpoint_pivos_cb, early_stop_pivos_cb],
)

final_pivos_path = os.path.join(pasta_base, f'Modelo_Pivos_{SAFRA_ALVO}_FINAL.keras')
modelo_pivos.save(final_pivos_path)
print(f"✅ Modelo auxiliar de pivôs salvo em: {final_pivos_path}")


## Exemplos positivos de pivô encontrados

Execute esta célula logo após o treinamento específico dos pivôs. Ela procura imagens com maior percentual de pixels positivos e mostra os exemplos onde há pivô no gabarito ou na predição da IA.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("--- EXEMPLOS POSITIVOS DE PIVÔ ENCONTRADOS ---")

pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'
KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
PIVO_FEATURE_BANDS = INPUT_BANDS + ['DELTA_NDVI', 'ABS_DELTA_NDVI', 'NDVI_MAX', 'NDVI_MEAN', 'COLHEITA_SIGNAL', 'SOLO_EXPOSTO_2']
LIMIAR_PIVO_EXEMPLO = 0.50
MIN_PIXELS_POSITIVOS = 0.005
MAX_EXEMPLOS_PIVOS = 8

caminho_modelo_pivos = os.path.join(pasta_base, f'Modelo_Pivos_{SAFRA_ALVO}_FINAL.keras')
if not os.path.exists(caminho_modelo_pivos):
    raise FileNotFoundError(f'Modelo de pivôs não encontrado: {caminho_modelo_pivos}. Execute a célula de treinamento de pivôs primeiro.')

modelo_pivos = tf.keras.models.load_model(caminho_modelo_pivos, compile=False)
print(f"✅ Modelo de pivôs carregado: {caminho_modelo_pivos}")

padroes_busca = [f'*{SAFRA_ALVO}*MASSIVE*tfrecord*', f'*MASSIVE*{SAFRA_ALVO}*tfrecord*', f'*{SAFRA_ALVO}*tfrecord*']
busca = []
for padrao in padroes_busca:
    busca = glob.glob(os.path.join(pasta_base, padrao))
    if busca:
        busca.sort(key=os.path.getmtime, reverse=True)
        break
if not busca:
    raise FileNotFoundError(f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}.')
caminho_arquivo = busca[0]


def parse_pivos_exemplos(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)
    r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
    delta_ndvi = ndvi2 - ndvi1
    image_stacked = tf.concat([
        r1, nir1, ndvi1, r2, nir2, ndvi2,
        delta_ndvi,
        tf.abs(delta_ndvi),
        tf.maximum(ndvi1, ndvi2),
        (ndvi1 + ndvi2) / 2.0,
        tf.nn.relu(ndvi1 - ndvi2),
        tf.cast(ndvi2 < 0.25, tf.float32),
    ], axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, tf.cast(lbl > 0.5, tf.float32)


dataset_exemplos = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(parse_pivos_exemplos).batch(16)
positivos = []
for imgs_batch, labels_batch in dataset_exemplos.take(20):
    preds_batch = modelo_pivos.predict(imgs_batch, verbose=0)
    pred_bin = (preds_batch[:, :, :, 0] >= LIMIAR_PIVO_EXEMPLO).astype(np.float32)
    label_np = labels_batch.numpy()[:, :, :, 0]
    for i in range(imgs_batch.shape[0]):
        frac_pred = float(np.mean(pred_bin[i]))
        frac_label = float(np.mean(label_np[i]))
        if frac_pred >= MIN_PIXELS_POSITIVOS or frac_label >= MIN_PIXELS_POSITIVOS:
            positivos.append((frac_pred + frac_label, imgs_batch[i].numpy(), label_np[i], preds_batch[i, :, :, 0], pred_bin[i], frac_pred, frac_label))

positivos = sorted(positivos, key=lambda item: item[0], reverse=True)[:MAX_EXEMPLOS_PIVOS]
print(f"⭕ Exemplos positivos encontrados: {len(positivos)}")

if not positivos:
    print("⚠️ Nenhum exemplo positivo foi encontrado nos primeiros lotes. Diminua MIN_PIXELS_POSITIVOS ou confira os rótulos label_chip.")
else:
    plt.figure(figsize=(18, 4 * len(positivos)))
    for linha, (_, img, label, prob, pred_bin, frac_pred, frac_label) in enumerate(positivos):
        plt.subplot(len(positivos), 4, linha * 4 + 1)
        plt.imshow(img[:, :, 2], cmap='RdYlGn', vmin=0, vmax=0.8)
        plt.axis('off')
        if linha == 0:
            plt.title('NDVI inicial')

        plt.subplot(len(positivos), 4, linha * 4 + 2)
        plt.imshow(label, cmap='binary_r', vmin=0, vmax=1)
        plt.axis('off')
        if linha == 0:
            plt.title('Pivô no gabarito')

        plt.subplot(len(positivos), 4, linha * 4 + 3)
        plt.imshow(prob, cmap='magma', vmin=0, vmax=1)
        plt.axis('off')
        if linha == 0:
            plt.title('Prob. IA pivô')

        plt.subplot(len(positivos), 4, linha * 4 + 4)
        plt.imshow(pred_bin, cmap='binary_r', vmin=0, vmax=1)
        plt.axis('off')
        plt.title(f'Resultado pivô\nIA={frac_pred*100:.2f}% | rótulo={frac_label*100:.2f}%')

    plt.tight_layout()
    exemplos_path = os.path.join(pasta_base, f'Exemplos_Positivos_Pivos_{SAFRA_ALVO}.png')
    plt.savefig(exemplos_path, dpi=150, bbox_inches='tight')
    print(f"🖼️ Exemplos positivos de pivô salvos em: {exemplos_path}")
    plt.show()


Passo 2: O Teste Visual da Safra 2025 (A Prova de Fogo)
Agora vamos carregar o modelo salvo para 2025 e aplicar em imagens TFRecord da mesma safra para verificar se ele desenha os pivôs.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
from google.colab import drive

# 1. Montar Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("--- INICIANDO PROVA REAL DA SAFRA 2025 COM FENOLOGIA ---")

# 2. Localizar o Modelo Salvo
pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
SAFRA_ALVO = '2025'
SAFRA_INICIO = '2024-12-01'
SAFRA_FIM = '2025-03-31'
DATA_1_DESC = 'janela inicial da safra (dezembro/2024)'
DATA_2_DESC = 'janela final da safra (março/2025)'
caminho_modelo = os.path.join(pasta_base, f'Modelo_UNet_Jussara_{SAFRA_ALVO}_FINAL_v2.keras')

if os.path.exists(caminho_modelo):
    print(f"✅ Arquivo do modelo encontrado: {caminho_modelo}")

    # CARREGAR O MODELO (O momento da verdade)
    try:
        model = tf.keras.models.load_model(caminho_modelo)
        print("✅ Modelo carregado na memória com sucesso!")
    except Exception as e:
        raise RuntimeError(f"❌ ERRO ao carregar modelo: {e}") from e
else:
    raise FileNotFoundError(f"❌ ERRO: o arquivo .keras da safra {SAFRA_ALVO} não foi encontrado em {caminho_modelo}.")

# 3. Carregar um pouco de dados da safra-alvo para testar
padroes_busca = [
    f'*{SAFRA_ALVO}*MASSIVE*tfrecord*',
    f'*MASSIVE*{SAFRA_ALVO}*tfrecord*',
    f'*{SAFRA_ALVO}*tfrecord*',
]

busca_dados = []
for padrao in padroes_busca:
    busca_dados = glob.glob(os.path.join(pasta_base, padrao))
    if busca_dados:
        busca_dados.sort(key=os.path.getmtime, reverse=True)
        break

if not busca_dados:
    raise FileNotFoundError(
        f'Nenhum TFRecord da safra {SAFRA_ALVO} encontrado em {pasta_base}. '
        'Confira se o arquivo contém 2025 no nome.'
    )

caminho_dados = busca_dados[0]
print(f"📂 Validando com dados da safra {SAFRA_ALVO}: {caminho_dados}")
print(f"🗓️ Janela fenológica configurada: {SAFRA_INICIO} até {SAFRA_FIM}")

KERNEL_SIZE = 128
READ_SIZE = 129
# _1 representa a janela inicial (dez/2024) e _2 representa a janela final (mar/2025).
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
USAR_FENOLOGIA = True
USAR_VARIACAO_PAISAGEM = True
USAR_AUGE_VIGOR = True
LIMIAR_ALTO_VIGOR_NDVI = 0.55
FENOLOGIA_BANDS = [
    'DELTA_NDVI',
    'ABS_DELTA_NDVI',
    'NDVI_MAX',
    'NDVI_MEAN',
    'PLANTIO_SIGNAL',
    'CERRADO_STABILITY',
]
AUGE_VIGOR_BANDS = [
    'VIGOR_MAX_NDVI',
    'VIGOR_PEAK_TIMING',
    'VIGOR_PEAK_CONFIDENCE',
    'VIGOR_ALTO_MASK',
]
PAISAGEM_BANDS = [
    'COLHEITA_SIGNAL',
    'SOLO_EXPOSTO_1',
    'SOLO_EXPOSTO_2',
    'SOLO_EXPOSTO_AUMENTO',
    'MUDANCA_PAISAGEM',
    'ALERTA_MUDANCA',
]
FEATURE_BANDS = (
    INPUT_BANDS
    + (FENOLOGIA_BANDS if USAR_FENOLOGIA else [])
    + (AUGE_VIGOR_BANDS if USAR_AUGE_VIGOR else [])
    + (PAISAGEM_BANDS if USAR_VARIACAO_PAISAGEM else [])
)

def adicionar_fenologia(inputs_list):
    r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
    delta_ndvi = ndvi2 - ndvi1
    abs_delta_ndvi = tf.abs(delta_ndvi)
    ndvi_max = tf.maximum(ndvi1, ndvi2)
    ndvi_mean = (ndvi1 + ndvi2) / 2.0
    plantio_signal = tf.nn.relu(delta_ndvi)
    cerrado_stability = ndvi_mean * (1.0 - tf.clip_by_value(abs_delta_ndvi, 0.0, 1.0))
    return tf.concat([
        delta_ndvi,
        abs_delta_ndvi,
        ndvi_max,
        ndvi_mean,
        plantio_signal,
        cerrado_stability,
    ], axis=-1)

def adicionar_auge_vigor(inputs_list):
    r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
    vigor_max_ndvi = tf.maximum(ndvi1, ndvi2)
    vigor_peak_timing = tf.cast(ndvi2 >= ndvi1, tf.float32)
    vigor_peak_confidence = tf.abs(ndvi2 - ndvi1)
    vigor_alto_mask = tf.cast(vigor_max_ndvi >= LIMIAR_ALTO_VIGOR_NDVI, tf.float32)
    return tf.concat([
        vigor_max_ndvi,
        vigor_peak_timing,
        vigor_peak_confidence,
        vigor_alto_mask,
    ], axis=-1)

def adicionar_variacao_paisagem(inputs_list):
    r1, nir1, ndvi1, r2, nir2, ndvi2 = inputs_list
    colheita_signal = tf.nn.relu(ndvi1 - ndvi2)
    solo_exposto_1 = tf.cast(ndvi1 < 0.25, tf.float32)
    solo_exposto_2 = tf.cast(ndvi2 < 0.25, tf.float32)
    solo_exposto_aumento = tf.nn.relu(solo_exposto_2 - solo_exposto_1)
    mudanca_paisagem = (tf.abs(r2 - r1) + tf.abs(nir2 - nir1) + tf.abs(ndvi2 - ndvi1)) / 3.0
    alerta_mudanca = tf.cast(
        (colheita_signal > 0.20) | (solo_exposto_aumento > 0.50) | (mudanca_paisagem > 0.20),
        tf.float32,
    )
    return tf.concat([
        colheita_signal,
        solo_exposto_1,
        solo_exposto_2,
        solo_exposto_aumento,
        mudanca_paisagem,
        alerta_mudanca,
    ], axis=-1)

def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        inputs_list.append(img)
    image_stacked = tf.concat(inputs_list, axis=-1)
    if USAR_FENOLOGIA:
        image_stacked = tf.concat([image_stacked, adicionar_fenologia(inputs_list)], axis=-1)
    if USAR_AUGE_VIGOR:
        image_stacked = tf.concat([image_stacked, adicionar_auge_vigor(inputs_list)], axis=-1)
    if USAR_VARIACAO_PAISAGEM:
        image_stacked = tf.concat([image_stacked, adicionar_variacao_paisagem(inputs_list)], axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    return image_stacked, lbl

# Pega apenas 1 lote de 10 imagens
dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP')
dataset = dataset.map(parse_fast).batch(10).take(1)

# 4. Gerar Previsões
print(f"🌱 Validando com {len(FEATURE_BANDS)} bandas de entrada: {FEATURE_BANDS}")
print("🔮 Gerando previsões com o modelo carregado...")
imgs, labels = next(iter(dataset))
preds = model.predict(imgs)

# Resultado final dos pivôs no lote visual: máscara binária da predição.
# Use este mapa para ver claramente onde a IA está marcando pivô.
LIMIAR_RESULTADO_PIVO = 0.50
resultado_pivos = (preds[:, :, :, 0] >= LIMIAR_RESULTADO_PIVO).astype(np.float32)

feature_index = {band: idx for idx, band in enumerate(FEATURE_BANDS)}
alertas = imgs[:, :, :, feature_index['ALERTA_MUDANCA']]
solo_exposto = imgs[:, :, :, feature_index['SOLO_EXPOSTO_2']]
colheita = imgs[:, :, :, feature_index['COLHEITA_SIGNAL']]
auge_vigor = imgs[:, :, :, feature_index['VIGOR_MAX_NDVI']]
pico_timing = imgs[:, :, :, feature_index['VIGOR_PEAK_TIMING']]
alto_vigor = imgs[:, :, :, feature_index['VIGOR_ALTO_MASK']]

print(f"🚨 Pixels com alerta no lote: {float(tf.reduce_mean(alertas).numpy()) * 100:.2f}%")
print(f"🌾 Sinal médio de possível colheita no lote: {float(tf.reduce_mean(colheita).numpy()):.4f}")
print(f"🟫 Pixels de solo exposto na janela final: {float(tf.reduce_mean(solo_exposto).numpy()) * 100:.2f}%")
print(f"🌿 Pixels em alto vigor no lote: {float(tf.reduce_mean(alto_vigor).numpy()) * 100:.2f}%")
print(f"📈 Pico médio de vigor mais próximo da janela final: {float(tf.reduce_mean(pico_timing).numpy()) * 100:.2f}%")
print(f"⭕ Pixels marcados como pivô pela IA: {float(np.mean(resultado_pivos)) * 100:.2f}% (limiar={LIMIAR_RESULTADO_PIVO:.2f})")

# 5. Visualizar
plt.figure(figsize=(28, 12))
print("\nLEGENDA: Satélite | Gabarito | Auge do vigor | Alertas de mudança/solo/colheita | Probabilidade da IA | Resultado pivôs")

for i in range(5): # Mostra 5 exemplos
    # Satélite (NDVI Safra)
    plt.subplot(5, 6, i*6 + 1)
    plt.imshow(imgs[i][:,:,2], cmap='RdYlGn', vmin=0, vmax=0.8)
    plt.axis('off')
    if i==0: plt.title(f'Satélite (NDVI {SAFRA_ALVO})')

    # Gabarito
    plt.subplot(5, 6, i*6 + 2)
    plt.imshow(labels[i][:,:,0], cmap='binary_r')
    plt.axis('off')
    if i==0: plt.title('Gabarito Real')

    # Auge do vigor
    plt.subplot(5, 6, i*6 + 3)
    plt.imshow(auge_vigor[i], cmap='YlGn', vmin=0, vmax=0.9)
    plt.axis('off')
    if i==0: plt.title('Auge do vigor')

    # Alertas de variação da paisagem
    plt.subplot(5, 6, i*6 + 4)
    plt.imshow(alertas[i], cmap='OrRd', vmin=0, vmax=1)
    plt.axis('off')
    if i==0: plt.title('Alerta paisagem')

    # Probabilidade da IA
    plt.subplot(5, 6, i*6 + 5)
    # Vmin/Vmax fixos para ver a confiança real
    plt.imshow(preds[i][:,:,0], cmap='magma', vmin=0, vmax=1)
    plt.axis('off')
    if i==0: plt.title('Prob. IA pivô')

    # Resultado final dos pivôs
    plt.subplot(5, 6, i*6 + 6)
    plt.imshow(resultado_pivos[i], cmap='binary_r', vmin=0, vmax=1)
    plt.axis('off')
    if i==0: plt.title('Resultado pivôs')

plt.tight_layout()
resultado_figura = os.path.join(pasta_base, f'Resultado_Pivos_Jussara_{SAFRA_ALVO}.png')
plt.savefig(resultado_figura, dpi=150, bbox_inches='tight')
print(f"🖼️ Figura com resultado dos pivôs salva em: {resultado_figura}")
plt.show()
